In [1]:
import json
import os
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
!source /home/jupyter/Mrigi/env.sh
hf_token = os.environ.get("HF_TOKEN")


In [3]:
# ---------- Step 1: Load and Process the JSON Records ----------
# Load JSON records from file. Right now it only has 100 or so papers
with open('filtered_records_v2.json', 'r') as file:
    filtered_records = json.load(file)

In [4]:
# Process each record: extract paragraphs with "Experimental" supersection,
# concatenate them, and append DOI info.
documents = []
metadata = []

In [5]:
for record in filtered_records:
    experimental_paragraphs = [
        para.get("text", "")
        for para in record.get("paragraphs", [])
        if para.get("supersection_name", "").strip() == "Experimental"
    ]
    
    # Only add papers that have experimental sections
    if experimental_paragraphs:
        combined_text = "\n\n".join(experimental_paragraphs)
        doi = record.get("doi", "Unknown DOI")
        combined_text += f"\n\nThis information is from DOI: {doi}"
        documents.append(combined_text)
        metadata.append({"doi": doi})

print(f"Total papers with experimental sections: {len(documents)}")

Total papers with experimental sections: 28


In [6]:
# for doc in documents:
#     print(doc)
#     print("-"*40)  # Optional: adds a visual separator between different papers


In [7]:
# ---------- Step 2: Create the Vector Database with FAISS and SciBERT ----------
# Use SciBERT for embeddings via HuggingFaceEmbeddings.
embeddings = HuggingFaceEmbeddings(model_name="allenai/scibert_scivocab_uncased")

No sentence-transformers model found with name /home/synthesisproject/.cache/torch/sentence_transformers/allenai_scibert_scivocab_uncased. Creating a new one with MEAN pooling.


In [8]:
# Build a FAISS vector store where each document represents a paper’s experimental section.
vector_db = FAISS.from_texts(documents, embeddings, metadatas=metadata)

DeferredCudaCallError: CUDA call failed lazily at initialization with error: device >= 0 && device < num_gpus INTERNAL ASSERT FAILED at "../aten/src/ATen/cuda/CUDAContext.cpp":50, please report a bug to PyTorch. 

CUDA call was originally invoked at:

['  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/runpy.py", line 197, in _run_module_as_main\n    return _run_code(code, main_globals, None,\n', '  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/runpy.py", line 87, in _run_code\n    exec(code, run_globals)\n', '  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/site-packages/ipykernel_launcher.py", line 17, in <module>\n    app.launch_new_instance()\n', '  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/site-packages/traitlets/config/application.py", line 1077, in launch_instance\n    app.start()\n', '  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/site-packages/ipykernel/kernelapp.py", lin

In [ ]:
# ---------- Step 3: Use the Vector DB in a RAG Pipeline ----------
# Define your query.
query = "How is Fe-CHA (Chabazite) synthesized?"


In [331]:
# Retrieve top k relevant documents (papers) from the vector database.
retrieved_docs = vector_db.similarity_search(query, k=2)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [332]:
# print(embeddings.embed_query("Silicalite-1 synthesis"))
# print(embeddings.embed_documents(["test text 1", "test text 2"]))


In [333]:
context_text

'The zeolites are made from basic silicate solution with organic templates. Low molecular weight silicate species such as Cab-O-Sil M5 (Cabot Corporation) or LUDOX HS 30 (DuPont), Fe(NO3)3 salt (Aldrich) and FeIII(acac)2 complex (Aldrich) and Al(NO3)3 salt (JT-Baker) are used as silicate, iron and aluminum precursors, LiOH or NaOH are added. The basic silicate solution can be added to an acidified ferric salt solution, to give an overall basic gel. The acidified ferric salt solution contains O-coordination complexes such as nitrates or acetylacetonates of iron. These form stable complexes at low pH, but slowly dissociate in a basic medium liberating the metal ions. During the dissociation the silica can bind to form the ferri-silicate gel thus, avoiding the precipitation of iron as hydroxides. Once the ferri-silicate gel is formed, precipitation of iron is avoided even at higher pH. The organic structure directing agents are 1-adamanteinium-trimethyl-ammonium-hydroxide (ATMA, CHA), tet

In [334]:
# Define a prompt template that instructs the LLM how to answer.
PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Provide which DOI the answer is retrieved from.
"""

In [335]:
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, question=query)

In [336]:
# ---------- Step 4: Generate an Answer Using an Open-Source LLM ----------
# Setup the open source LLM using Mistral (ensure you have the model or access to it)
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False, use_auth_token=hf_token)

/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/site-packages/transformers/models/auto/tokenization_auto.py:671: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, use_auth_token=hf_token, torch_dtype=torch.float16)

/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/site-packages/transformers/models/auto/auto_factory.py:472: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [339]:
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=100000)

In [340]:
# Wrap the Hugging Face pipeline with LangChain's HuggingFacePipeline interface.
llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [341]:
# Generate the answer based on the prompt that includes retrieved context.
response_text = llm(prompt)
print("Response:")
print(response_text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Response:

Answer: Fe-CHA (Chabazite) is synthesized by direct synthesis using a basic silicate solution and an organic structure directing agent (SDA). The basic silicate solution is prepared by mixing low molecular weight silica (LUDOX HS-30) with iron(III) nitrate (Fe(NO3)3) and sodium hydroxide (NaOH). The organic SDA used in the synthesis is 1-adamanteinium-trimethyl-ammonium-hydroxide (ATMA, CHA). The mixture is heated at 100°C at autogeneous pressure for 7 days to form Fe-CHA.

Reference(s):
DOI: 10.1016/j.apcatb.2013.09.049


In [342]:
!nvidia-smi

Fri Mar 21 16:00:25 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 495.29.05    Driver Version: 495.29.05    CUDA Version: 11.5     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A5000    Off  | 00000000:31:00.0 Off |                  Off |
| 30%   38C    P2    91W / 230W |  22717MiB / 24256MiB |      4%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A5000    Off  | 00000000:4B:00.0 Off |                  Off |
| 30%   